# 06 — Phoenix Land Cover Segmentation

Evaluates the DynamicWorld-architecture segmentation outputs on Phoenix composites.

**Sections**
1. RGB + segmentation overlay for 6 sample months
2. Confusion matrix vs NLCD 2021 (if available)
3. Built-class accuracy — headline metric

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import rasterio
import rioxarray
from dotenv import load_dotenv

load_dotenv()

from urbangrowth.config import data_path, get_pipeline
from urbangrowth.processing.segmentation import CLASSES, CLASS_COLORS, N_CLASSES

CITY = 'phoenix'
pipe = get_pipeline()

COMPOSITE_DIR = data_path(pipe['processed_data_subdirs']['composites'],  CITY)
SEG_DIR       = data_path(pipe['processed_data_subdirs']['land_cover'],  CITY)

# Build colour LUT (index → RGB float [0,1])
COLOR_LUT = np.zeros((N_CLASSES, 3), dtype=np.float32)
for i, cls in enumerate(CLASSES):
    COLOR_LUT[i] = [c / 255.0 for c in CLASS_COLORS[cls]]

all_class_tifs = sorted(SEG_DIR.glob('*_class.tif'))
print(f'Composite dir : {COMPOSITE_DIR}')
print(f'Segmentation  : {SEG_DIR}')
print(f'Class rasters : {len(all_class_tifs)}')

## 1. RGB + Segmentation Overlay — 6 Sample Months

In [ ]:
def stretch(arr, lo=2, hi=98):
    v = arr[np.isfinite(arr)]
    if v.size == 0:
        return arr
    vmin, vmax = np.percentile(v, [lo, hi])
    return np.clip((arr - vmin) / (vmax - vmin + 1e-9), 0, 1)


def class_to_rgb(class_map, alpha=0.55):
    """Convert uint8 class raster to RGBA overlay."""
    H, W = class_map.shape
    rgba = np.zeros((H, W, 4), dtype=np.float32)
    for i in range(N_CLASSES):
        mask = class_map == i
        rgba[mask, :3] = COLOR_LUT[i]
        rgba[mask,  3] = alpha
    return rgba


def month_label(p: Path) -> str:
    return p.name[:7]  # 'YYYY-MM'


if not all_class_tifs:
    print('No segmentation results yet.  Run:  ug segment run --city phoenix')
else:
    step = max(1, len(all_class_tifs) // 6)
    samples = all_class_tifs[::step][:6]

    fig, axes = plt.subplots(len(samples), 2, figsize=(14, 4.5 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]

    for row, cls_tif in enumerate(samples):
        label = month_label(cls_tif)
        comp_tif = COMPOSITE_DIR / f'{label}.tif'

        # Class map
        with rasterio.open(cls_tif) as src:
            class_map = src.read(1)

        # RGB from composite (B04/B03/B02 = bands 3/2/1 in 0-based → bands 3,2,1 in 1-based)
        if comp_tif.exists():
            with rasterio.open(comp_tif) as src:
                # composite band order: B02=1, B03=2, B04=3, B08=4, B11=5, B12=6, ...
                rgb_raw = src.read([3, 2, 1]).astype(float)  # B04, B03, B02
            rgb = np.stack([stretch(rgb_raw[c]) for c in range(3)], axis=-1)
        else:
            rgb = np.ones((*class_map.shape, 3), dtype=np.float32) * 0.5

        overlay = class_to_rgb(class_map)

        # RGB panel
        axes[row, 0].imshow(rgb, interpolation='bilinear')
        axes[row, 0].set_title(f'{label} — RGB', fontsize=10)
        axes[row, 0].axis('off')

        # Segmentation overlay
        axes[row, 1].imshow(rgb, interpolation='bilinear')
        axes[row, 1].imshow(overlay, interpolation='nearest')
        axes[row, 1].set_title(f'{label} — Land Cover', fontsize=10)
        axes[row, 1].axis('off')

    # Legend
    patches = [
        mpatches.Patch(color=tuple(c / 255 for c in CLASS_COLORS[cls]), label=cls)
        for cls in CLASSES
    ]
    fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, -0.02))
    fig.suptitle('Phoenix DynamicWorld Land Cover — Sample Months', fontsize=13)
    plt.tight_layout()
    plt.show()

## 2. Confusion Matrix vs NLCD 2021

NLCD 2021 must be clipped to the Phoenix bbox and placed at:
`{data_root}/raw/nlcd/phoenix_nlcd2021.tif`

NLCD → DynamicWorld class mapping used:
| NLCD value | NLCD name | DW class |
|---|---|---|
| 11 | Open Water | water |
| 21,22,23,24 | Developed | built |
| 31 | Barren | bare |
| 41,42,43 | Forest | trees |
| 52 | Shrub/Scrub | shrub_scrub |
| 71 | Herbaceous | grass |
| 81 | Hay/Pasture | crops |
| 82 | Cultivated Crops | crops |
| 90 | Woody Wetlands | flooded_veg |
| 95 | Emergent Herb. Wetlands | flooded_veg |

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

NLCD_TO_DW: dict[int, int] = {
    11: 0,   # water
    21: 6, 22: 6, 23: 6, 24: 6,  # built
    31: 7,   # bare
    41: 1, 42: 1, 43: 1,          # trees
    52: 5,   # shrub_scrub
    71: 2,   # grass
    81: 4, 82: 4,                  # crops
    90: 3,   # flooded_veg
    95: 3,   # flooded_veg
}

# Try to load NLCD reference for a mid-archive month
nlcd_path = data_path('raw', 'nlcd') / 'phoenix_nlcd2021.tif'

if not nlcd_path.exists():
    print(f'NLCD 2021 reference not found at: {nlcd_path}')
    print('Download from https://www.mrlc.gov/data and clip to Phoenix bbox.')
    print('Skipping confusion matrix.')
    NLCD_AVAILABLE = False
else:
    NLCD_AVAILABLE = True

if NLCD_AVAILABLE and all_class_tifs:
    # Use the most recent available class raster
    ref_tif = all_class_tifs[-1]
    label = month_label(ref_tif)

    with rasterio.open(ref_tif) as src:
        pred_full = src.read(1)
        transform = src.transform
        crs = src.crs

    # Reproject NLCD to match the prediction raster
    import rasterio.warp as rwarp
    with rasterio.open(nlcd_path) as nlcd_src:
        nlcd_reprojected = np.zeros_like(pred_full, dtype=np.uint8)
        rwarp.reproject(
            source=rasterio.band(nlcd_src, 1),
            destination=nlcd_reprojected,
            src_transform=nlcd_src.transform,
            src_crs=nlcd_src.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=rwarp.Resampling.nearest,
        )

    # Map NLCD to DW classes; ignore unmapped pixels
    ref_dw = np.full_like(nlcd_reprojected, 255, dtype=np.uint8)
    for nlcd_val, dw_cls in NLCD_TO_DW.items():
        ref_dw[nlcd_reprojected == nlcd_val] = dw_cls

    valid = ref_dw < 255
    y_true = ref_dw[valid].astype(int)
    y_pred = pred_full[valid].astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(CLASSES, fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('NLCD Reference')
    ax.set_title(f'Confusion matrix vs NLCD 2021 — {label} composite', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, label='Row-normalised accuracy')

    for r in range(N_CLASSES):
        for c in range(N_CLASSES):
            v = cm_norm[r, c]
            ax.text(c, r, f'{v:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if v > 0.5 else 'black')
    plt.tight_layout()
    plt.show()

    oa = cm.diagonal().sum() / cm.sum()
    print(f'\nOverall accuracy vs NLCD: {oa:.3f}')
    print()
    print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))

## 3. Built-Class Accuracy — Headline Metric

The built class (index 6) is the primary signal for urban growth detection.
We report precision, recall, F1 against NLCD, plus the built-class pixel
fraction across all available months.

In [ ]:
BUILT_IDX = CLASSES.index('built')

# ── Confusion-matrix based metrics vs NLCD ────────────────────────────────
if NLCD_AVAILABLE and 'y_true' in dir():
    tp = int(((y_true == BUILT_IDX) & (y_pred == BUILT_IDX)).sum())
    fp = int(((y_true != BUILT_IDX) & (y_pred == BUILT_IDX)).sum())
    fn = int(((y_true == BUILT_IDX) & (y_pred != BUILT_IDX)).sum())

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)

    print('Built-class performance vs NLCD 2021')
    print(f'  Precision : {precision:.3f}')
    print(f'  Recall    : {recall:.3f}')
    print(f'  F1        : {f1:.3f}')
    print(f'  TP        : {tp:,}   FP: {fp:,}   FN: {fn:,}')
    print()

# ── Built fraction time series ────────────────────────────────────────────
if all_class_tifs:
    records = []
    for cls_tif in all_class_tifs:
        with rasterio.open(cls_tif) as src:
            cm = src.read(1)
        total = cm.size
        built_pct = float((cm == BUILT_IDX).sum()) / total * 100
        records.append({'month': month_label(cls_tif), 'built_pct': round(built_pct, 2)})

    built_df = pd.DataFrame(records).set_index('month')

    fig, ax = plt.subplots(figsize=(16, 3.5))
    ax.plot(range(len(built_df)), built_df['built_pct'], marker='o', ms=3,
            color=tuple(c / 255 for c in CLASS_COLORS['built']), linewidth=1.5)

    tick_positions = list(range(0, len(built_df), max(1, len(built_df) // 16)))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([built_df.index[i] for i in tick_positions], rotation=45, ha='right')
    ax.set_ylabel('Built pixel fraction (%)')
    ax.set_title('Phoenix — built-class coverage over time')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('Built fraction summary:')
    print(built_df['built_pct'].describe().round(3).to_string())
    first_val = built_df['built_pct'].iloc[0]
    last_val  = built_df['built_pct'].iloc[-1]
    print(f'\nChange {built_df.index[0]} → {built_df.index[-1]}: '
          f'{last_val - first_val:+.2f} pp')

In [ ]:
# Class distribution across all months
if all_class_tifs:
    class_counts = np.zeros(N_CLASSES, dtype=np.int64)
    for cls_tif in all_class_tifs:
        with rasterio.open(cls_tif) as src:
            cm = src.read(1)
        for i in range(N_CLASSES):
            class_counts[i] += int((cm == i).sum())

    total_px = class_counts.sum()
    fractions = class_counts / total_px * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    colors_plot = [tuple(c / 255 for c in CLASS_COLORS[cls]) for cls in CLASSES]
    bars = ax.barh(CLASSES, fractions, color=colors_plot)
    ax.set_xlabel('Mean pixel fraction (%)')
    ax.set_title('Phoenix — time-averaged land cover distribution')
    for bar, frac in zip(bars, fractions):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                f'{frac:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()